# Frozen transformer surprise prototype

This notebook monitors projected intermediate activations with a tiny online temporal expectation model. It reports error and experience separately; early observations remain `unknown`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo_dir = Path("/content/neural-expectation")
if not (repo_dir / "pyproject.toml").exists():
    subprocess.run(
        ["git", "clone", "https://github.com/sebastiantiesmeyer/neural-expectation.git", str(repo_dir)],
        check=True,
    )
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"], check=True)

import matplotlib.pyplot as plt
import numpy as np
import torch
from transformer_surprise.activations import project_selected_layers
from transformer_surprise.config import load_config
from transformer_surprise.model import FrozenTransformer
from transformer_surprise.projection import FixedRandomProjection
from transformer_surprise.state_space import RLSStateSpace

In [ ]:
config = load_config("configs/default.yaml")
encoder = FrozenTransformer.from_pretrained(config.model_name, config.layers, config.device, config.dtype)
projection = FixedRandomProjection(encoder.model.config.hidden_size, config.projection_dim, config.projection_seed)
print(f"device: {encoder.device}; layers: {config.layers}")

In [ ]:
sentences = [
    "The cat sat on the mat.",
    "The dog sat by the door.",
    "The cat slept on the chair.",
    "The cat sat on the mat.",
    "The dog sat by the door.",
    "The cta sat on the mat.",
    "The cat transformed into a neutron star.",
]
encoded = encoder.tokenizer(sentences, return_tensors="pt", padding=True, truncation=True)
hidden = encoder.hidden_states(sentences)
projected = project_selected_layers(hidden, projection, encoded["attention_mask"])
print({layer: tuple(values.shape) for layer, values in projected.items()})

In [ ]:
models = {layer: RLSStateSpace(config.projection_dim, config.sidecar.forgetting_factor, config.sidecar.regularization) for layer in config.layers}
records = []
for index in range(len(sentences) - 1):
    for layer, sidecar in models.items():
        result = sidecar.update(projected[layer][index].numpy(), projected[layer][index + 1].numpy())
        records.append({"step": index, "layer": layer, **result.as_dict()})
        print(f"step={index} layer={layer} status={result.status} error={result.error:.3f} confidence={result.confidence:.2f}")

In [ ]:
for layer in config.layers:
    points = [record for record in records if record["layer"] == layer]
    plt.plot([point["step"] for point in points], [point["error"] for point in points], marker="o", label=f"layer {layer}")
plt.xlabel("transition")
plt.ylabel("L2 prediction residual")
plt.title("Layer-wise online prediction error")
plt.legend()
plt.show()